In [1]:
import ipywidgets as widgets
print(widgets.__version__)

8.1.8


In [2]:
!pip install nltk scikit-learn pdfminer.six python-docx ipywidgets

import re
import string
import nltk
import numpy as np
import ipywidgets as widgets
import io
import base64
from IPython.display import display, HTML, clear_output
from sklearn.feature_extraction.text import TfidfVectorizer   # ← TF-IDF, better than CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pdfminer.high_level import extract_text_to_fp
from pdfminer.layout import LAParams
from docx import Document

In [13]:
import io
import re
import nltk
from docx import Document
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML


In [14]:
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOP_WORDS = set(stopwords.words('english'))

SKILLS_DB = {
    "Languages": {
        "Python":            ["python"],
        "Java":              ["java"],
        "C++":               ["c++", "cpp"],
        "JavaScript":        ["javascript", "js", "node"],
        "R":                 [" r ", "r language", "r programming"],
        "SQL":               ["sql", "mysql", "postgresql", "sqlite"],
        "Scala":             ["scala"],
        "Go":                [" go ", "golang"],
        "TypeScript":        ["typescript", "ts"],
    },
    "ML / AI": {
        "Machine Learning":  ["machine learning", " ml ", "ml model", "supervised", "unsupervised"],
        "Deep Learning":     ["deep learning", " dl ", "neural network", "cnn", "rnn", "lstm"],
        "NLP":               ["nlp", "natural language", "text mining", "text classification"],
        "Computer Vision":   ["computer vision", " cv ", "image processing", "object detection"],
        "LLM / GenAI":       ["llm", "large language model", "generative ai", "gpt", "bert", "transformer"],
        "Reinforcement RL":  ["reinforcement learning", " rl "],
    },
    "Frameworks": {
        "TensorFlow":        ["tensorflow", "tf"],
        "PyTorch":           ["pytorch", "torch"],
        "Scikit-learn":      ["scikit", "sklearn", "scikit-learn"],
        "OpenCV":            ["opencv", "cv2"],
        "Flask":             ["flask"],
        "FastAPI":           ["fastapi"],
        "Django":            ["django"],
        "Streamlit":         ["streamlit"],
        "HuggingFace":       ["huggingface", "transformers library", "hugging face"],
        "XGBoost":           ["xgboost", "lightgbm", "gradient boost"],
    },
    "Data & Analytics": {
        "Pandas":            ["pandas"],
        "NumPy":             ["numpy"],
        "Spark":             ["spark", "pyspark"],
        "Tableau":           ["tableau"],
        "Power BI":          ["power bi", "powerbi"],
        "Excel":             ["excel", "spreadsheet"],
        "Data Analysis":     ["data analysis", "data analytics", "eda", "exploratory"],
        "Data Visualization":["data visualization", "data viz", "matplotlib", "seaborn", "plotly"],
        "ETL":               ["etl", "data pipeline", "data engineering"],
    },
    "Cloud / MLOps": {
        "AWS":               ["aws", "amazon web services", "ec2", "s3", "sagemaker"],
        "Azure":             ["azure", "microsoft azure"],
        "GCP":               ["gcp", "google cloud", "bigquery"],
        "Docker":            ["docker", "containeriz"],
        "Kubernetes":        ["kubernetes", "k8s"],
        "MLflow":            ["mlflow", "ml flow"],
        "Git":               ["git", "github", "gitlab", "version control"],
        "Linux":             ["linux", "unix", "bash", "shell"],
        "CI/CD":             ["ci/cd", "cicd", "devops", "jenkins", "github actions"],
    },
    "Soft Skills": {
        "Communication":     ["communication", "communicat", "presentation"],
        "Leadership":        ["leadership", "led team", "managed team", "lead"],
        "Agile/Scrum":       ["agile", "scrum", "sprint", "kanban", "jira"],
        "Problem Solving":   ["problem solving", "analytical", "critical thinking"],
        "Teamwork":          ["teamwork", "collaboration", "cross-functional"],
    }
}

# ── Text extraction ──────────────────────────────────────────
def extract_pdf_text(file_bytes: bytes) -> str:
    from pdfminer.high_level import extract_text_to_fp
    from pdfminer.layout import LAParams
    output = io.StringIO()
    extract_text_to_fp(io.BytesIO(file_bytes), output, laparams=LAParams())
    return output.getvalue()

def extract_docx_text(file_bytes: bytes) -> str:
    doc = Document(io.BytesIO(file_bytes))
    return "\n".join([p.text for p in doc.paragraphs])

def extract_text_from_bytes(file_bytes: bytes, filename: str) -> str:
    ext = filename.rsplit(".", 1)[-1].lower()
    if ext == "pdf":
        return extract_pdf_text(file_bytes)
    elif ext == "docx":
        return extract_docx_text(file_bytes)
    return ""

# ── Improved Skill matching (Handles plural and spacing) ──────────────────
def find_skills_in_text(text: str) -> dict:
    text = f" {text.lower()} " # Add padding for boundary matching
    found = {}
    for category, skills in SKILLS_DB.items():
        matched = []
        for skill_name, aliases in skills.items():
            for alias in aliases:
                # Use regex to find exact word matches only
                pattern = r'(?i)\b' + re.escape(alias.strip()) + r'(s|es)?\b'
                if re.search(pattern, text):
                    matched.append(skill_name)
                    break
        if matched:
            found[category] = matched
    return found

# ── Improved Scoring (More balanced weights) ──────────────────────────────
def calculate_score(resume_raw: str, jd_raw: str) -> dict:
    res_low = resume_raw.lower()
    jd_low  = jd_raw.lower()

    # --- 1. Skill Overlap (The Core) ---
    res_skills = set(s for cat in find_skills_in_text(res_low).values() for s in cat)
    jd_skills = set(s for cat in find_skills_in_text(jd_low).values() for s in cat)
    
    if not jd_skills:
        skill_score = 70.0 
    else:
        # We set the target to 70% of JD skills to be more realistic
        overlap = len(res_skills & jd_skills)
        target = len(jd_skills)
        skill_score = min((overlap / (target * 0.7)) * 100, 100) 

    # --- 2. Semantic Similarity (The 'Vibe' Check) ---
    try:
        # Using char n-grams to catch partial matches like "Develop" vs "Developer"
        vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), stop_words='english')
        matrix = vec.fit_transform([res_low, jd_low])
        semantic_sim = cosine_similarity(matrix)[0][1]
        
        # Boost: Normalizing strict math to human-friendly 40-80% range
        semantic_score = min(semantic_sim * 250, 100)
    except:
        semantic_score = 0

    # --- 3. Final Weighted Score ---
    # 70% Skills + 30% Semantic Similarity
    final = (skill_score * 0.7) + (semantic_score * 0.3)
    
    # --- 4. Top Skill Bonus ---
    # If you match the first 3 skills in the JD, you get a 15 point boost
    top_jd_skills = list(jd_skills)[:3]
    if top_jd_skills and all(s in res_skills for s in top_jd_skills):
        final += 15

    final = round(min(final, 98.5), 1) 

    return {
        "final": final,
        "skill_score": round(skill_score, 1),
        "kw_score": round(semantic_score, 1),
        "tfidf_score": round(semantic_score, 1),
        "res_skills": res_skills,
        "jd_skills": jd_skills,
    }

# ── Missing skills & feedback ────────────────────────────────
def get_missing(res_skills: set, jd_skills: set) -> list:
    return sorted(jd_skills - res_skills)

def get_feedback(score: float, missing: list) -> tuple:
    if score >= 75:
        return "Strong match",   "✅", "#28a745", "Well aligned! Tailor your summary to echo the JD's exact wording."
    elif score >= 50:
        return "Moderate match", "⚠️", "#fd7e14", f"Good base. Try adding: {', '.join(missing[:4]) or 'more JD keywords'}."
    elif score >= 30:
        return "Weak match",     "🔶", "#dc3545", f"Significant gaps. Focus on: {', '.join(missing[:4]) or 'upskilling'}."
    else:
        return "Low match",      "❌", "#c62828", "Resume may not be targeting this role. Review the JD carefully."

# ── HTML report ──────────────────────────────────────────────
def build_report_html(scores, res_skill_dict, missing, level, emoji, color, advice):
    score = scores['final']

    breakdown = f"""
    <div style="margin-top:12px;font-size:12px;color:#555">
      <div style="display:flex;align-items:center;gap:8px;margin:4px 0">
        <span style="width:130px">Skill overlap (50%)</span>
        <div style="flex:1;background:#f0f0f0;border-radius:4px;height:7px">
          <div style="background:#4caf50;width:{scores['skill_score']}%;height:7px;border-radius:4px"></div>
        </div>
        <span style="width:40px;text-align:right">{scores['skill_score']}%</span>
      </div>
      <div style="display:flex;align-items:center;gap:8px;margin:4px 0">
        <span style="width:130px">Keyword match (30%)</span>
        <div style="flex:1;background:#f0f0f0;border-radius:4px;height:7px">
          <div style="background:#2196f3;width:{scores['kw_score']}%;height:7px;border-radius:4px"></div>
        </div>
        <span style="width:40px;text-align:right">{scores['kw_score']}%</span>
      </div>
      <div style="display:flex;align-items:center;gap:8px;margin:4px 0">
        <span style="width:130px">TF-IDF cosine (20%)</span>
        <div style="flex:1;background:#f0f0f0;border-radius:4px;height:7px">
          <div style="background:#9c27b0;width:{scores['tfidf_score']}%;height:7px;border-radius:4px"></div>
        </div>
        <span style="width:40px;text-align:right">{scores['tfidf_score']}%</span>
      </div>
    </div>"""

    skill_html = ""
    for cat, skills in res_skill_dict.items():
        pills = "".join(f'<span style="background:#e8f5e9;color:#2e7d32;padding:3px 10px;border-radius:999px;font-size:12px;margin:3px;display:inline-block">{s}</span>' for s in skills)
        skill_html += f'<p style="font-size:12px;color:#888;margin:8px 0 3px;font-weight:600">{cat}</p>{pills}'

    miss_pills = "".join(
        f'<span style="background:#fdecea;color:#c62828;padding:3px 10px;border-radius:999px;font-size:12px;margin:3px;display:inline-block">{s}</span>'
        for s in missing
    ) or '<span style="color:#888;font-size:13px">✅ None — great coverage!</span>'

    return f"""
    <div style="font-family:sans-serif;max-width:720px;margin:16px 0">
      <div style="background:#fff;border:1px solid #e0e0e0;border-radius:12px;padding:24px;margin-bottom:14px">
        <div style="display:flex;align-items:center;justify-content:space-between">
          <span style="font-size:18px;font-weight:600">Overall Match Score</span>
          <span style="font-size:36px;font-weight:700;color:{color}">{score}%</span>
        </div>
        <div style="background:#f0f0f0;border-radius:999px;height:12px;margin:10px 0">
          <div style="background:{color};width:{score}%;height:12px;border-radius:999px"></div>
        </div>
        <div style="padding:10px 14px;background:#f9f9f9;border-radius:8px;border-left:4px solid {color}">
          {emoji} <b>{level}</b> — {advice}
        </div>
        <details style="margin-top:12px;cursor:pointer">
          <summary style="font-size:13px;color:#888">Show score breakdown</summary>
          {breakdown}
        </details>
      </div>
      <div style="background:#fff;border:1px solid #e0e0e0;border-radius:12px;padding:24px;margin-bottom:14px">
        <div style="font-size:16px;font-weight:600;margin-bottom:10px">🧠 Skills Found in Your Resume</div>
        {skill_html or '<span style="color:#888;font-size:13px">No known skills detected.</span>'}
      </div>
      <div style="background:#fff;border:1px solid #e0e0e0;border-radius:12px;padding:24px">
        <div style="font-size:16px;font-weight:600;margin-bottom:10px">⚠️ Skills in JD but Missing from Resume</div>
        {miss_pills}
      </div>
    </div>
    """

In [15]:
uploader = widgets.FileUpload(accept='.pdf,.docx', multiple=False, description='📄 Upload Resume')
jd_box   = widgets.Textarea(placeholder='Paste Job Description here...',
                             layout=widgets.Layout(width='100%', height='140px'))
run_btn  = widgets.Button(description='🔍 Analyze Resume', button_style='primary',
                          layout=widgets.Layout(margin='10px 0'))
out      = widgets.Output()

display(widgets.HTML("<h2>🤖 AI Resume Analyzer</h2>"))
display(widgets.HTML("<p><b>Step 1</b> → Upload resume &nbsp;|&nbsp; <b>Step 2</b> → Paste JD &nbsp;|&nbsp; <b>Step 3</b> → Analyze</p>"))
display(uploader)
display(widgets.HTML("<br><b>Job Description:</b>"))
display(jd_box)
display(run_btn)
display(out)

def read_uploaded_file(uploader_widget):
    val = uploader_widget.value
    if isinstance(val, tuple):
        if len(val) == 0:
            return None, None
        f = val[0]
        return f['name'], bytes(f['content'])
    elif isinstance(val, dict):
        if len(val) == 0:
            return None, None
        filename = list(val.keys())[0]
        return filename, bytes(val[filename]['content'])
    else:
        print(f"⚠️ Unknown uploader.value type: {type(val)}")
        return None, None

def on_analyze(b):
    with out:
        clear_output()

        filename, content = read_uploaded_file(uploader)
        if filename is None:
            print("❌ No file detected.")
            print("   → Make sure you see '(1)' next to the Upload button after selecting a file.")
            print("   → Try: Kernel → Restart & Run All, then upload again.")
            return

        print(f"📂 File loaded: {filename}  ({len(content)/1024:.1f} KB)")

        jd_raw = jd_box.value.strip()
        if not jd_raw:
            print("❌ Please paste a job description.")
            return

        print("⏳ Analyzing...")

        try:
            raw_text = extract_text_from_bytes(content, filename)
        except Exception as e:
            print(f"❌ Text extraction failed: {e}")
            import traceback; traceback.print_exc()
            return

        if not raw_text.strip():
            print("❌ Resume text is empty.")
            print("   → If it's a scanned PDF (image-only), text extraction won't work.")
            return

        scores     = calculate_score(raw_text, jd_raw)           # ← fixed: was "score"
        res_skills = find_skills_in_text(raw_text.lower())
        missing    = get_missing(scores['res_skills'], scores['jd_skills'])
        level, emoji, color, advice = get_feedback(scores['final'], missing)
        display(HTML(build_report_html(scores, res_skills, missing, level, emoji, color, advice)))

run_btn.on_click(on_analyze)

HTML(value='<h2>🤖 AI Resume Analyzer</h2>')

HTML(value='<p><b>Step 1</b> → Upload resume &nbsp;|&nbsp; <b>Step 2</b> → Paste JD &nbsp;|&nbsp; <b>Step 3</b…

FileUpload(value=(), accept='.pdf,.docx', description='📄 Upload Resume')

HTML(value='<br><b>Job Description:</b>')

Textarea(value='', layout=Layout(height='140px', width='100%'), placeholder='Paste Job Description here...')

Button(button_style='primary', description='🔍 Analyze Resume', layout=Layout(margin='10px 0'), style=ButtonSty…

Output()